In [1]:
import numpy as np
import pandas as pd

from exerpy import ExergyAnalysis
from exerpy.analyses import _load_json

In [2]:
def diff_between_simulators(testcase):
    simulator_results = []
    for path in testcase.values():
        contents = _load_json(path)
        if "settings" not in contents:
            contents["settings"] = {}

        simulator_results += [
            ExergyAnalysis.from_json(path, **contents["settings"])
        ]

    sim1 = simulator_results[0]
    sim2 = simulator_results[1]

    columns = ["m", "p", "T"]
    if sim1.chemExLib is not None:
        columns.append("e_CH")
    if sim1.split_physical_exergy:
        columns.append("e_M")
        columns.append("e_T")
    else:
        columns.append("e_PH")

    df_sim1 = pd.DataFrame.from_dict(
        sim1._connection_data, orient="index"
    ).sort_index()[columns].dropna(how="all")
    df_sim2 = pd.DataFrame.from_dict(
        sim2._connection_data, orient="index"
    ).sort_index()[columns].dropna(how="all")

    overlapping_index = list(
        set(df_sim1.index.tolist()) & set(df_sim2.index.tolist())
    )
    df_sim1 = df_sim1.loc[overlapping_index].round(6)
    df_sim2 = df_sim2.loc[overlapping_index].round(6)

    # inf means that sim2 has 0 value, comparison does not make sense there
    # and sometimes there seem to be NaN values in the dataframes, those are
    # removed as well
    return (
        (df_sim1 - df_sim2) / df_sim2
    ).abs().replace(np.inf, 0).fillna(0)

In [3]:
testcase = {
    "ebsilon": "hp_ebs.json",
    "tespy": "hp_tespy.json"
}
diff_between_simulators(testcase)

,m,p,T,e_PH
12,0.000002,9.971237e-10,1.134148e-05,0.000227
32,0.000079,0.000000e+00,3.625946e-09,0.000145
11,0.000002,0.000000e+00,0.000000e+00,0.000000
30,0.000079,0.000000e+00,0.000000e+00,0.000126
23,0.000000,0.000000e+00,0.000000e+00,0.000003
31,0.000079,0.000000e+00,0.000000e+00,0.000126
13,0.000002,0.000000e+00,0.000000e+00,0.000266
22,0.000000,0.000000e+00,0.000000e+00,0.000009
21,0.000000,0.000000e+00,0.000000e+00,0.000009
33,0.000079,1.454546e-10,1.154332e-05,0.000003


In [4]:
testcase = {
    "tespy": "hp_tespy.json",
    "aspen": "hp_aspen.json"
}
diff_between_simulators(testcase)

,m,p,T,e_PH
12,0.000483,0.0,9.652330e-07,2.210114e-04
32,0.000093,0.0,2.538162e-08,1.529795e-04
11,0.000483,0.0,0.000000e+00,0.000000e+00
23,0.000000,0.0,2.543558e-09,6.815802e-07
31,0.000093,0.0,0.000000e+00,1.327341e-04
13,0.000483,0.0,0.000000e+00,6.316969e-03
22,0.000000,0.0,0.000000e+00,6.664713e-07
21,0.000000,0.0,0.000000e+00,6.677163e-07
33,0.000093,0.0,1.231574e-06,7.624343e-06
34,0.000093,0.0,4.539902e-05,7.303922e-05


In [5]:
testcase = {
    "ebsilon": "hp_ebs.json",
    "aspen": "hp_aspen.json"
}
diff_between_simulators(testcase)

,m,p,T,e_PH
12,0.000485,9.971237e-10,1.037626e-05,0.000006
32,0.000013,0.000000e+00,2.900757e-08,0.000008
11,0.000485,0.000000e+00,0.000000e+00,0.000000
23,0.000000,0.000000e+00,2.543558e-09,0.000002
31,0.000013,0.000000e+00,0.000000e+00,0.000007
13,0.000485,0.000000e+00,0.000000e+00,0.006581
22,0.000000,0.000000e+00,0.000000e+00,0.000009
21,0.000000,0.000000e+00,0.000000e+00,0.000009
33,0.000013,1.454546e-10,1.031176e-05,0.000005
34,0.000013,0.000000e+00,5.471011e-06,0.000011


In [6]:
# Component results stored by the example scripts hp_ebs.py, hp_tespy.py and hp_aspen.py
results = {
    simulator: pd.read_csv(f"hp_components_{simulator.lower()}.csv", index_col=0).set_index("Component")
    for simulator in ("Ebsilon", "TESPy", "Aspen")
}

# Components the simulators model differently are compared as one group
groups = {
    "PUMP-MOTOR": "MOT3",  # Aspen name of the pump motor
}

# Exergy destruction in kW of the components found in at least two models, and the deviations from Ebsilon
E_D = (
    pd.DataFrame({simulator: df["E_D [kW]"] for simulator, df in results.items()})
    .drop("TOT")
    .rename(index=groups)
    .groupby(level=0)
    .sum(min_count=1)
    .dropna(thresh=2)
)
reference = E_D["Ebsilon"].where(E_D["Ebsilon"].abs() > 1e-3)  # no relative deviation below 1 W
for simulator in ("TESPy", "Aspen"):
    E_D[f"{simulator} vs Ebsilon [%]"] = (E_D[simulator] / reference - 1) * 100
E_D.sort_values("Ebsilon", ascending=False).round(2)

,Ebsilon,TESPy,Aspen,TESPy vs Ebsilon [%],Aspen vs Ebsilon [%]
Component,,,,,
EVA,647.20,647.06,647.47,-0.02,0.04
VAL,144.57,144.61,144.57,0.02,-0.00
COMP,125.81,125.82,125.81,0.00,-0.00
FAN,104.88,104.86,104.93,-0.02,0.05
COND,91.49,91.48,91.50,-0.01,0.00
MOT2,13.61,13.62,13.41,0.00,-1.50
MOT1,10.74,10.74,10.59,-0.02,-1.45
PUMP,0.01,0.01,0.01,-0.21,-0.21
MOT3,0.00,0.00,0.00,NaN,NaN


In [7]:
# Results of the whole plant
columns = ["E_F [kW]", "E_P [kW]", "E_D [kW]", "E_L [kW]", "epsilon [%]"]
pd.DataFrame({simulator: df.loc["TOT", columns] for simulator, df in results.items()}).round(3)

,Ebsilon,TESPy,Aspen
E_F [kW],1624.046,1623.907,1624.018
E_P [kW],483.918,483.918,483.918
E_D [kW],1138.332,1138.192,1138.292
E_L [kW],1.796,1.796,1.808
epsilon [%],29.797,29.800,29.798
